# SigAlg's `L2` class

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `L2` class in SigAlg represents the *$L^2$-Hilbert space* of square-integrable random variables on a probability space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/l2/#sigalg.l2.L2).

## Mathematical definition

Let $(\Omega, \mathcal{F}, P)$ be a probability space. The *$L^2$-space* (or *$L^2$-Hilbert space*) associated with this probability space is the set

$$
L^2(\Omega, \mathcal{F}, P) = \left\{ X: \Omega \to \mathbb{R} \mid X \text{ is } \mathcal{F}\text{-measurable and } \int_\Omega X^2 \, dP < \infty \right\}.
$$

This is a vector space under pointwise addition and scalar multiplication. It becomes a *Hilbert space* when equipped with the inner product

$$
\langle X, Y \rangle \stackrel{\text{def}}{=} \int_\Omega XY \, dP = E(XY),
$$

which induces the *$L^2$-norm*

$$
\|X\| \stackrel{\text{def}}{=} \sqrt{\langle X, X \rangle} = \sqrt{E(X^2)},
$$

and the *$L^2$-metric*

$$
d(X,Y) \stackrel{\text{def}}{=} \|X - Y\| = \sqrt{E\left[(X-Y)^2\right]}.
$$

In the case that $\Omega$ is finite (as it always is in SigAlg), any $\mathcal{F}$-measurable random variable automatically satisfies $E(X^2) < \infty$, so $L^2(\Omega, \mathcal{F}, P)$ is simply the set of all $\mathcal{F}$-measurable random variables. Moreover, the space has an *orthonormal basis* consisting of the normalized indicator functions

$$
\phi_A = \frac{I_A}{\|I_A\|} = \frac{I_A}{\sqrt{P(A)}},
$$

where $A$ ranges over all atoms of $\mathcal{F}$ with nonzero probability. Any $X \in L^2(\Omega, \mathcal{F}, P)$ can thus be written as a *Fourier series*

$$
X = \sum_{A} \langle X, \phi_A \rangle \phi_A,
$$

where the sum extends over all atoms with nonzero probability.

## API examples



### Creating L2 spaces

#### From individual components

We begin by defining a sample space $\Omega = \{0,1,2,3\}$, a $\sigma$-algebra $\mathcal{F}$ on $\Omega$ with two atoms, and a probability measure $P$.

In [1]:
from sigalg.core import ProbabilityMeasure, SampleSpace, SigmaAlgebra

Omega = SampleSpace().from_sequence(size=4)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.45,
        3: 0.3,
    }
)

Create an `L2` space from these components.

In [2]:
from sigalg.l2 import L2

H = L2(sample_space=Omega, sigma_algebra=F, probability_measure=P, name="H")
print(H)

H = L2(Omega, F, P)

* Sample space 'Omega':
[0, 1, 2, 3]

* Sigma algebra 'F':
        atom ID
sample         
0             0
1             1
2             0
3             1

* Probability measure 'P':
        probability
sample             
0              0.10
1              0.15
2              0.45
3              0.30


####From a probability space

If we already have a `ProbabilitySpace` object that combines $(\Omega, \mathcal{F}, P)$, we can create an `L2` space directly from it.

In [ ]:
from sigalg.core import ProbabilitySpace

prob_space = ProbabilitySpace(sample_space=Omega, sigma_algebra=F, probability_measure=P)

H2 = L2(sample_space=prob_space.sample_space, sigma_algebra=prob_space.sigma_algebra, probability_measure=prob_space.probability_measure)
print(H2)

### Properties and the orthonormal basis

The `L2` space has several important properties. The `dim` property gives the dimension of the space, which equals the number of atoms of $\mathcal{F}$ with nonzero probability.

In [3]:
print(f"Dimension of H: {H.dim}")

Dimension of H: 2


The `basis` property returns a dictionary containing the orthonormal basis vectors $\phi_A = I_A / \|I_A\|$ for each atom $A$ with nonzero probability.

In [4]:
basis = H.basis
print("Orthonormal basis:")
for atom_id, phi in basis.items():
    print(f"  Atom {atom_id}: {phi}")

Orthonormal basis:
  Atom 0: Random variable '0':
             0
sample        
0       1.3484
1       0.0000
2       1.3484
3       0.0000
  Atom 1: Random variable '1':
               1
sample          
0       0.000000
1       1.490712
2       0.000000
3       1.490712


We can verify that these basis vectors are orthonormal by computing their inner products using the `inner` method. For details on inner products, see the dedicated [`inner` notebook](inner.ipynb).

In [5]:
print("Verifying orthonormality:")
for i, phi_i in basis.items():
    for j, phi_j in basis.items():
        inner_prod = H.inner(phi_i, phi_j)
        print(f"  ⟨φ_{i}, φ_{j}⟩ = {inner_prod:.6f}")

Verifying orthonormality:
  ⟨φ_0, φ_0⟩ = 1.000000
  ⟨φ_0, φ_1⟩ = 0.000000
  ⟨φ_1, φ_0⟩ = 0.000000
  ⟨φ_1, φ_1⟩ = 1.000000


The inner products are $1$ when $i=j$ (norm equals 1) and $0$ when $i \neq j$ (orthogonality), confirming the basis is orthonormal.

### Containment checking

We can check if a random variable is in the $L^2$ space using the `in` operator. A random variable is in $L^2(\Omega, \mathcal{F}, P)$ if and only if it is $\mathcal{F}$-measurable.

In [6]:
from sigalg.core import RandomVariable

# Create an F-measurable random variable (constant on atoms of F)
X = RandomVariable(domain=Omega, name="X").from_dict(
    {
        0: -1,
        1: 3,
        2: -1,
        3: 3,
    }
)

print(f"X in H: {X in H}")

X in H: True


Create a finer $\sigma$-algebra $\mathcal{G}$ that contains $\mathcal{F}$ as a sub-$\sigma$-algebra.

In [7]:
G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 1,
        2: 2,
        3: 3,
    }
)

# Create a G-measurable random variable that is not F-measurable
Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: 2,
        2: 3,
        3: 4,
    }
)

print(f"Y in H: {Y in H}")

Y in H: False


Since $Y$ is not constant on the atoms of $\mathcal{F}$, it is not $\mathcal{F}$-measurable and hence not in $L^2(\Omega, \mathcal{F}, P)$.

### Fourier coefficients

Any random variable in the $L^2$ space can be expressed as a Fourier series in terms of the orthonormal basis. The `fourier_coefficients` method computes these coefficients.

In [8]:
coeffs = H.fourier_coefficients(X)
print("Fourier coefficients:")
for atom_id, coeff in coeffs.items():
    print(f"  Atom {atom_id}: {coeff:.6f}")

Fourier coefficients:
  Atom 0: -0.741620
  Atom 1: 2.012461


We can reconstruct $X$ from its Fourier coefficients by computing $X = \sum_A \langle X, \phi_A \rangle \phi_A$.

In [9]:
# Reconstruct X from Fourier series
X_reconstructed = sum(coeffs[i] * basis[i] for i in coeffs.keys())

print("Original X:")
print(X)
print("\nReconstructed X:")
print(X_reconstructed)
print(f"\nAre they equal? {X == X_reconstructed}")

Original X:
Random variable 'X':
        X
sample   
0      -1
1       3
2      -1
3       3

Reconstructed X:
Random variable '((0+(-0.7416198487095663*0))+(2.0124611797498106*1))':
        ((0+(-0.7416198487095663*0))+(2.0124611797498106*1))
sample                                                      
0                                                    -1.0   
1                                                     3.0   
2                                                    -1.0   
3                                                     3.0   

Are they equal? True


### Related methods

The `L2` class provides several key methods for working with the Hilbert space structure. Each has its own dedicated notebook with detailed examples:

- [**`inner`**](inner.ipynb) - Compute the inner product $\langle X, Y \rangle = E(XY)$ of two random variables
- [**`norm`**](norm.ipynb) - Compute the $L^2$-norm $\|X\| = \sqrt{E(X^2)}$ of a random variable
- [**`metric`**](metric.ipynb) - Compute the $L^2$-distance $d(X,Y) = \|X-Y\|$ between random variables
- [**`proj`**](proj.ipynb) - Compute the orthogonal projection of a random variable onto a subspace

These methods form the foundation for understanding conditional expectation, variance decomposition, and other probabilistic concepts in terms of Hilbert space geometry.